In [1]:
import torch

In [2]:
# load the names dataset from file
from pathlib import Path

data_dir = Path.cwd() / ".." / ".." / "data"


def load_names(path: Path) -> list[str]:
    with path.open("r") as f:
        return f.read().splitlines()


words = load_names(data_dir / "names.txt")
words[:4]

['emma', 'olivia', 'ava', 'isabella']

In [3]:
from makemore.vocab import Vocab
import string

# build the vocabulary
vocab = Vocab.from_words([string.ascii_lowercase])
len(vocab)

27

In [4]:
# build the dataset
import random
from makemore.data import make_dataset

BLOCK_SIZE = 3


def split(
    words: list[str],
    vocab: Vocab,
    *,
    block_size: int,
    splits: tuple[float, float, float],
    seed: int
) -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
]:
    """Split the dataset"""
    assert sum(v for v in splits) == 1.0, "broken precondition"

    random.seed(seed)
    random.shuffle(words)

    n1 = int(splits[0] * len(words))
    n2 = int((splits[0] + splits[1]) * len(words))

    X_train, Y_train = make_dataset(words[:n1], vocab, block_size)
    X_dev, Y_dev = make_dataset(words[n1:n2], vocab, block_size)
    X_test, Y_test = make_dataset(words[n2:], vocab, block_size)

    return X_train, Y_train, X_dev, Y_dev, X_test, Y_test


X_train, Y_train, X_dev, Y_dev, X_test, Y_test = split(
    words, vocab, block_size=BLOCK_SIZE, splits=(0.8, 0.1, 0.1), seed=42
)
print(X_train.shape, Y_train.shape)
print(X_dev.shape, Y_dev.shape)
print(X_test.shape, Y_test.shape)

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [38]:
# the character embedding dimension
embedding_dimension = 10
# the number of neurons in the hidden layer
hidden_layer_size = 200

g = torch.Generator().manual_seed(2147483647)

# embedding layer
C = torch.randn(
    (len(vocab), embedding_dimension),
    generator=g,
)
# hidden layer
kaiming_init_scale = (5 / 3) / ((BLOCK_SIZE * embedding_dimension) ** 0.5)
W1 = (
    torch.randn(
        (
            BLOCK_SIZE * embedding_dimension,
            hidden_layer_size,
        ),
        generator=g,
    )
    * kaiming_init_scale
)
b1 = torch.randn(hidden_layer_size, generator=g) * 0.01
# output layer
W2 = (
    torch.randn(
        (hidden_layer_size, len(vocab)),
        generator=g,
    )
    * 0.01
)
b2 = (
    torch.randn(
        len(
            vocab,
        ),
        generator=g,
    )
    * 0.0
)

parameters = [C, W1, b1, W2, b2]

print(f"kaiming init scale = {kaiming_init_scale}")

n_parameters = sum(p.nelement() for p in parameters)
print(f"total parameters = {n_parameters}")

for p in parameters:
    p.requires_grad = True

kaiming init scale = 0.3042903097250923
total parameters = 11897


In [39]:
import torch.nn.functional as F

max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
    # minibatch construct
    ix = torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
    Xb, Yb = X_train[ix], Y_train[ix]

    # --- forward pass

    # embed the characters into vectors
    emb = C[Xb]
    # concatenate the vectors
    embcat = emb.view(emb.shape[0], -1)
    # hidden layer pre-activation
    hpreact = embcat @ W1 + b1
    # non-linearity (complete hidden layer)
    h = torch.tanh(hpreact)
    # output layer
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)  # loss function

    # --- backward pass

    for p in parameters:
        p.grad = None
    loss.backward()

    # --- update

    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad  # type: ignore

    # --- track stats

    if i % 10000 == 0:
        print(f"{i:7d}/{max_steps:7d}: {loss.item():.4f}")
    lossi.append(loss.log10().item())

      0/ 200000: 3.3179


  10000/ 200000: 2.1910
  20000/ 200000: 2.3270
  30000/ 200000: 2.5396
  40000/ 200000: 1.9468
  50000/ 200000: 2.3331
  60000/ 200000: 2.3852
  70000/ 200000: 2.1173
  80000/ 200000: 2.3159
  90000/ 200000: 2.2010
 100000/ 200000: 1.8591
 110000/ 200000: 2.0881
 120000/ 200000: 1.9389
 130000/ 200000: 2.3913
 140000/ 200000: 2.0949
 150000/ 200000: 2.1458
 160000/ 200000: 1.7824
 170000/ 200000: 1.7249
 180000/ 200000: 1.9752
 190000/ 200000: 1.8614


In [40]:
@torch.no_grad()
def split_loss(split) -> float:
    x, y = {
        "train": (X_train, Y_train),
        "val": (X_dev, Y_dev),
        "test": (X_test, Y_test),
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    return loss.item()


print(f"train = {split_loss('train')}")
print(f"val   = {split_loss('val')}")

train = 2.0376644134521484
val   = 2.106989622116089


In [ ]:
import matplotlib.pyplot as plt

plt.plot(lossi)